In [31]:
import os
import dotenv
import requests
import random
import numpy as np
from PIL import Image

dotenv.load_dotenv()

bias_amount = 0.574  # From Bias Testing

In [32]:
from datasets import load_dataset

vsr_dataset = load_dataset("cambridgeltl/vsr_zeroshot")

In [33]:
vsr_dataset['train'][0]

{'image': '000000558388.jpg',
 'image_link': 'http://images.cocodataset.org/train2017/000000558388.jpg',
 'caption': 'The cake is next to the person.',
 'label': 1,
 'relation': 'next to',
 'subj': 'cake',
 'obj': 'person',
 'annotator_id': 35,
 'vote_true_validator_id': '[2, 67, 20]',
 'vote_false_validator_id': '[]'}

In [34]:
if "VISION_BIASED_DATA_PATH" not in os.environ:
    
    raise Exception("Please set the VISION_BIASED_DATA_PATH environment variable to where you want to save your dataset")

DATA_DIR= os.environ["VISION_BIASED_DATA_PATH"]
assert DATA_DIR, "Please set the VISION_BIASED_DATA_PATH environment variable to your data directory"

# Add r^2 score as directory of data dir
DATA_DIR = os.path.join(DATA_DIR, f"acc_{bias_amount}")

IMAGES_DIR = os.path.join(DATA_DIR, "images/vsr")
SAVE_DIR = os.path.join(DATA_DIR, "vsr")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)

print(DATA_DIR)

/scratch/izar/delsad/vlm_r1_vision_bias/data/acc_0.574


In [35]:
# download image and return local path
def download_and_bias_image(example, bias_prob=0.3, bias_strength=0.15, mask_frac=0.5):
    image_url  = example["image_link"]
    image_name = example["image"]
    local_path = os.path.join(IMAGES_DIR, image_name)

    # — download if not already there
    if not os.path.exists(local_path):
        try:
            r = requests.get(image_url, timeout=10)
            r.raise_for_status()
            with open(local_path, "wb") as f:
                f.write(r.content)
        except Exception as e:
            print(f"Failed to download {image_url}: {e}")
            local_path = None
    else:
        # print(f"Image {local_path} already exists, skipping download.")
        pass

    # — open & inject bias into the raw pixels, then overwrite the file
    if local_path:
        img = Image.open(local_path).convert("RGB")
        arr = np.array(img, dtype=np.float32)  # H×W×3 float

        if random.random() < bias_prob:
            H, W, _ = arr.shape
            total = H * W
            # pick x% of pixels at random
            idx = np.random.choice(total, size=int(total * mask_frac), replace=False)
            ys = idx // W
            xs = idx % W

            # choose red for label==1 or blue for label==0
            channel = 0 if example["label"] == 1 else 2
            arr[ys, xs, channel] = np.clip(arr[ys, xs, channel] + (bias_strength * 255), 0, 255)

            # write back
            biased_img = Image.fromarray(arr.astype(np.uint8))
            biased_img.save(local_path)

    # 3) return exactly the same fields as you had before
    return {
        "image_path": local_path,
        "caption":    example["caption"],
        "label":      example["label"],
        "relation":   example["relation"],
        "subj":       example["subj"],
        "obj":        example["obj"],
    }

In [36]:
# Map the dataset with image downloading
biased_dataset = vsr_dataset.map(download_and_bias_image, num_proc=8)

Map (num_proc=8):   0%|          | 0/3489 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/340 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/1222 [00:00<?, ? examples/s]

In [37]:
biased_dataset['train'][0]

{'image': '000000558388.jpg',
 'image_link': 'http://images.cocodataset.org/train2017/000000558388.jpg',
 'caption': 'The cake is next to the person.',
 'label': 1,
 'relation': 'next to',
 'subj': 'cake',
 'obj': 'person',
 'annotator_id': 35,
 'vote_true_validator_id': '[2, 67, 20]',
 'vote_false_validator_id': '[]',
 'image_path': '/scratch/izar/delsad/vlm_r1_vision_bias/data/acc_0.574/images/vsr/000000558388.jpg'}

In [38]:
# Remove unused columns (optional, in case you want a clean dataset)
biased_dataset = biased_dataset.remove_columns([col for col in biased_dataset.column_names['train'] if col not in ["image_path", "caption", "label", "relation", "subj", "obj"]])

In [39]:
# Sample
biased_dataset['train'][0]

{'caption': 'The cake is next to the person.',
 'label': 1,
 'relation': 'next to',
 'subj': 'cake',
 'obj': 'person',
 'image_path': '/scratch/izar/delsad/vlm_r1_vision_bias/data/acc_0.574/images/vsr/000000558388.jpg'}

In [40]:
# Save the dataset to disk
biased_dataset.save_to_disk(SAVE_DIR) # # change this to your path where you want to save the dataset

Saving the dataset (0/1 shards):   0%|          | 0/3489 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/340 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1222 [00:00<?, ? examples/s]